In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data,AmlCompute
from azure.ai.ml.constants import AssetTypes


In [2]:
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

Found the config file in: ./config.json


In [13]:
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential

# This will trigger a browser pop-up to log into your Azure account
credential = InteractiveBrowserCredential()

# Initialize MLClient
ml_client = MLClient.from_config(credential=credential)

Found the config file in: ./config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [138]:
compute_name = "batch-cluster"

In [15]:
try:
    compute_target = ml.client.compute.get(compute_name)
    print(f"Found existing compute target: {compute_name}")
except Exception as e:
    compute_target = AmlCompute(name=compute_name, 
                                size="STANDARD_D2_V2", 
                                type="AmlCompute",
                                min_instances=0,
                                max_instances=4,
                                idle_time_before_scale_down=120
                                )


In [16]:
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential

# Explicitly target your subscription's correct tenant ID
tenant_id = "32631e6c-2bce-4d34-8fbf-dcd501a37c39"

credential = InteractiveBrowserCredential(tenant_id=tenant_id)

# Initialize MLClient
ml_client = MLClient.from_config(credential=credential)
print("Successfully authenticated and connected to workspace!")

Found the config file in: ./config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Successfully authenticated and connected to workspace!


In [7]:
ml_client.compute.begin_create_or_update(compute_target).result()

AmlCompute({'type': 'amlcompute', 'created_on': '2026-10-14T07:46:32.4833115+00:00', 'provisioning_state': <ProvisioningState.SUCCEEDED: 'Succeeded'>, 'provisioning_errors': None, 'name': 'batch-cluster', 'description': None, 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/7ec95194-0f00-495d-aaa5-b1be9a0cd09e/resourceGroups/Leadscoringdeployement/providers/Microsoft.MachineLearningServices/workspaces/LeadScoring/computes/batch-cluster', 'Resource__source_path': '', 'base_path': '/Users/ram/Desktop/Machine Learning Models/Leadscoring', 'creation_context': None, 'resource_id': None, 'location': 'eastus2', 'size': 'STANDARD_D2_V2', 'min_instances': 0, 'max_instances': 4, 'idle_time_before_scale_down': 120.0, 'identity': None, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.entities._compute.compute.NetworkSettings object at 0x11665ed30>, 'tier': 'dedicated', 'enable_node_public_ip': True, 'subnet': None})

In [32]:
## creating inference.py
import os
import joblib
import pandas as pd
def init():
    """Loads the model into memory once when the batch worker starts."""
    global model
    # AZUREML_MODEL_DIR points to your registered model directory
    model_dir = os.getenv("AZUREML_MODEL_DIR")
    model_path = os.path.join(model_dir, "model.joblib")
    model = joblib.load(model_path)

def run(mini_batch):
    """Processes incoming mini-batches of data files in parallel."""
    results = []
    for file_path in mini_batch:
        # Read each input data file (e.g., CSV)
        data = pd.read_csv(file_path)
        
        # Make predictions using your trained model
        predictions = model.predict(data)
        
        # Append predictions back to data frame
        df_out = data.copy()
        df_out["prediction"] = predictions
        results.append(df_out)
        
    return pd.concat(results, ignore_index=True)


In [33]:
from azure.ai.ml.entities import BatchEndpoint, BatchDeployment

In [17]:
endpoint_name = "lead-scoring-batch-endpoint"
deployment_name = "v1"

In [18]:
endpoint = BatchEndpoint(
    name=endpoint_name,
    description="Batch scoring endpoint for lead scoring",
)
ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

NameError: name 'BatchEndpoint' is not defined

In [41]:
from azure.ai.ml.entities import ModelBatchDeployment

endpoint_name = "lead-scoring-batch-endpoint"
deployment_name = "v1"

# Use ModelBatchDeployment instead of BatchDeployment
deployment = ModelBatchDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model="lead-scoring-model:1",  # Your registered model and version
    scoring_script="inference_azure.py",
    code_path="./",              # Path to the folder containing inference.py
    environment="AzureML-sklearn-1.5-ubuntu22.04-py310:1",
    compute="batch-cluster",      # Your compute cluster name
    instance_count=2,
    mini_batch_size=2,
    output_action="append_row",
)

# Submit the deployment to Azure
ml_client.batch_deployments.begin_create_or_update(deployment).result()
print(f"Batch endpoint '{endpoint_name}' and deployment '{deployment_name}' are ready!")

Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/Users/ram/Desktop/Machine Learning Models/Leadscoring' 'https://leadscoring2548308906.blob.core.windows.net/133af24a-810e-484d-aebe-60dec5eb4cd2-o759dfv8l9b275jhtn81yrwhik/Leadscoring' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading Leadscoring (1460.38 MBs):  40%|███▉      | 583430505/1460375904 [10:31<15:49, 923980.34it/s]   


KeyboardInterrupt: 

In [ ]:

###authenticate your Python script directly to your Azure Storage Account so your code can read, write, and manage files or datasets in the cloud without needing interactive browser sign-ins.

x₹
import os


# Paste your actual connection string from the Azure Portal here
os.environ[
    "AZURE_STORAGE_CONNECTION_STRING"
] = "DefaultEndpointsProtocol=https;AccountName=leadscoring2548308906;AccountKey=Au2LFku8dwCUXdQ9AsHsS6agRxooYDdxK4M5Fx1Hs34b8CDKbC9Wb7VYSVm8/J0w6BJjQHvAseKQ+AStPcqDAw==;EndpointSuffix=core.windows.net"

In [4]:
import os
import pandas as pd
import joblib
import pyodbc
from azure.storage.blob import BlobServiceClient


In [5]:
#### load the model from Azure Blob Storage
def load_model():
    connect_str = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    blob_service_client = BlobServiceClient.from_connection_string(connect_str)


    blob_client = blob_service_client.get_blob_client(
        container="ml-model-container-leadscoring", blob="models/lead_scoring/best_xg_model.joblib"
    )
    model_path = "best_xg_model.joblib"
    with open(model_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    print(f"Model downloaded to {model_path}")
    return joblib.load(model_path)

In [6]:
model = load_model()

Model downloaded to best_xg_model.joblib


In [54]:
model

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None,
              eval_metrics='logloss', feature_types=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [47]:
from sklearn.metrics import classification_report

In [77]:
### fetc the unscored leads
def fetch_unscored_leads():
    connect_str = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    blob_service_client = BlobServiceClient.from_connection_string(connect_str)

    blob_client = blob_service_client.get_blob_client(
    container="ml-model-container-leadscoring", blob="rawdata/X_test.csv")
    file_path = "X_test.csv"
    with open(file_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    print(f"Unscored leads data downloaded to {file_path}")

    return pd.read_csv(file_path)

In [133]:
X_test = fetch_unscored_leads()
X_test

Unscored leads data downloaded to X_test.csv


,Page Views Per Visit SQRT,TotalVisits SQRT,Total Time Spent on Website SQRT,Lead Number,Lead Origin_API,Lead Origin_Landing Page Submission,Lead Origin_Lead Add Form,Lead Origin_Lead Import,Lead Origin_Quick Add Form,Lead Source_Click2call,...,Last Notable Activity_Form Submitted on Website,Last Notable Activity_Had a Phone Conversation,Last Notable Activity_Modified,Last Notable Activity_Olark Chat Conversation,Last Notable Activity_Page Visited on Website,Last Notable Activity_Resubscribed to emails,Last Notable Activity_SMS Sent,Last Notable Activity_Unreachable,Last Notable Activity_Unsubscribed,Last Notable Activity_View in browser link Clicked
0,1.634013,2.828427,15.874508,615582,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1.414214,1.414214,30.479501,588939,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,1.000000,1.000000,1.414214,621242,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,1.414214,1.414214,17.972201,589803,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1.732051,1.732051,14.177447,651441,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2583,2.080865,3.605551,30.886890,588188,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2584,1.732051,1.732051,7.937254,586447,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2585,1.414214,1.414214,29.274562,635012,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2586,0.000000,0.000000,0.000000,610648,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [129]:

X_test['Total Time Spent on Website SQRT'].isnull().sum()

np.int64(0)

Page Views Per Visit SQRT
TotalVisits SQRT
Total Time Spent on Website SQRT
Lead Number
Lead Origin_API
Lead Origin_Landing Page Submission
Lead Origin_Lead Add Form
Lead Origin_Lead Import
Lead Origin_Quick Add Form
Lead Source_Click2call
Lead Source_Direct Traffic
Lead Source_Google
Lead Source_Live Chat
Lead Source_NC_EDM
Lead Source_Olark Chat
Lead Source_Organic Search
Lead Source_Pay per Click Ads
Lead Source_Press_Release
Lead Source_Reference
Lead Source_Referral Sites
Lead Source_Social Media
Lead Source_WeLearn
Lead Source_Welingak Website
Lead Source_bing
Lead Source_blog
Lead Source_testone
Do Not Email_No
Do Not Email_Yes
Last Activity_Approached upfront
Last Activity_Converted to Lead
Last Activity_Email Bounced
Last Activity_Email Link Clicked
Last Activity_Email Marked Spam
Last Activity_Email Opened
Last Activity_Email Received
Last Activity_Form Submitted on Website
Last Activity_Had a Phone Conversation
Last Activity_Not Defined
Last Activity_Olark Chat Conversation


In [128]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2588 entries, 0 to 2587
Columns: 155 entries, Page Views Per Visit SQRT to prediction
dtypes: float64(3), int64(152)
memory usage: 3.1 MB


In [87]:
def score_leads(model, leads_df):
    predictions = model.predict(leads_df)
    leads_df["prediction"] = predictions
    return predictions

In [88]:
Y_pred = score_leads(model, X_test)

In [89]:
Y_pred

array([0, 0, 0, ..., 1, 0, 1])

In [90]:
Y_test = pd.read_csv("y_test.csv")

In [91]:
y_true = Y_test['Converted']

In [92]:
y_true


0       1
1       0
2       0
3       0
4       0
       ..
2583    1
2584    0
2585    1
2586    0
2587    1
Name: Converted, Length: 2588, dtype: int64

In [116]:
accuracy = accuracy_score(y_true, Y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 86.67%


In [117]:
bool_cols = X_test.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X_test = X_test.copy()
    X_test[bool_cols] = X_test[bool_cols].astype(int)

In [118]:
non_numeric_cols = X_test.select_dtypes(include=['object', 'category', 'string']).columns
if len(non_numeric_cols) > 0:
    raise ValueError(
        f"Found unencoded non-numeric columns: {list(non_numeric_cols)}. "
        "Please encode these columns into numeric formats before calculating SHAP values."
    )

In [119]:
X_test_numeric = X_test.select_dtypes(include=['number'])

In [105]:
X_test_numeric 

,Page Views Per Visit SQRT,TotalVisits SQRT,Total Time Spent on Website SQRT,Lead Number,Lead Origin_API,Lead Origin_Landing Page Submission,Lead Origin_Lead Add Form,Lead Origin_Lead Import,Lead Origin_Quick Add Form,Lead Source_Click2call,...,Last Notable Activity_Had a Phone Conversation,Last Notable Activity_Modified,Last Notable Activity_Olark Chat Conversation,Last Notable Activity_Page Visited on Website,Last Notable Activity_Resubscribed to emails,Last Notable Activity_SMS Sent,Last Notable Activity_Unreachable,Last Notable Activity_Unsubscribed,Last Notable Activity_View in browser link Clicked,prediction
0,1.634013,2.828427,15.874508,615582,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1.414214,1.414214,30.479501,588939,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,1.000000,1.000000,1.414214,621242,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,1.414214,1.414214,17.972201,589803,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1.732051,1.732051,14.177447,651441,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2583,2.080865,3.605551,30.886890,588188,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
2584,1.732051,1.732051,7.937254,586447,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2585,1.414214,1.414214,29.274562,635012,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
2586,0.000000,0.000000,0.000000,610648,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [106]:
X_clean = X_test.select_dtypes(include=["number"]).copy()
X_clean = X_clean.fillna(
    0
)

In [108]:
import numpy as np

In [109]:
X_mat = X_clean.to_numpy(dtype=np.float32)

In [110]:
import shap

In [112]:
explainer = shap.TreeExplainer(model)

In [120]:
shap_values = explainer.shap_values(X_test_numeric)

XGBoostError: [15:42:23] /Users/runner/work/xgboost/xgboost/src/c_api/c_api_utils.h:129: Check failed: std::accumulate(shape.cbegin(), shape.cend(), static_cast<bst_ulong>(1), std::multiplies<>{}) == chunksize * rows (403728 vs. 401140) : 
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x0000000136f7c428 dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x0000000136ffdda4 xgboost::CalcPredictShape(bool, xgboost::PredictionType, unsigned long, unsigned long, unsigned long, unsigned long, unsigned long, std::__1::vector<unsigned long long, std::__1::allocator<unsigned long long>>*, unsigned long long*) + 1224
  [bt] (2) 3   libxgboost.dylib                    0x0000000136ffd1c4 XGBoosterPredictFromDMatrix + 1228
  [bt] (3) 4   libffi.dylib                        0x000000019d667050 ffi_call_SYSV + 80
  [bt] (4) 5   libffi.dylib                        0x000000019d6705b8 ffi_call_int + 1220
  [bt] (5) 6   _ctypes.cpython-39-d

In [ ]:
best_model = model.best_estimator_
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values()

In [75]:
from sklearn.metrics import accuracy_score, classification_report

In [19]:
import pandas as pd


def load_test_data():
  # Define the file paths on your Mac
  x_test_path = (
      "/Users/ram/Desktop/Machine Learning Models/Leadscoring/X_test.csv"
  )
  y_test_path = (
      "/Users/ram/Desktop/Machine Learning Models/Leadscoring/Y_test.csv"
  )

  # Read the CSV files into pandas DataFrames
  X_test = pd.read_csv(x_test_path)
  Y_test = pd.read_csv(y_test_path)

  return X_test, Y_test


# --- How to use it in your script or notebook ---
if __name__ == "__main__":
  # 1. Load your test data locally
  X_test, Y_test = load_test_data()

  # 2. Load your model (either from local or blob storage)
  # model = joblib.load("model.joblib")

  # 3. Make predictions
  # predictions = model.predict(X_test)
  # print(f"Successfully generated predictions for {len(X_test)} rows.")

In [21]:
Y_test

,Converted
0,1
1,0
2,0
3,0
4,0
...,...
2583,1
2584,0
2585,1
2586,0


In [23]:
def update_lead_scores(df_scored):
    conn_str = os.getenv("AZURE_SQL_CONNECTION_STRING")
    with pyodbc.connect(conn_str) as conn:
        cursor = conn.cursor()
        for _, row in df_scored.iterrows():
            cursor.execute(
                "UPDATE Leads SET lead_score = ?, scored_at = GETDATE() WHERE lead_id = ?",
                float(row['predicted_score']),
                row['lead_id']
            )
        conn.commit()

In [ ]:
if __name__ == "__main__":
    model = load_model()
    leads_df = X_test
    
    if not leads_df.empty:
        features = leads_df[['feature_1', 'feature_2', 'feature_3']]
        # Assuming classification probability or regression score
        leads_df['predicted_score'] = model.predict_proba(features)[:, 1] 
        update_lead_scores(leads_df)
        print(f"Successfully scored {len(leads_df)} leads.")
    else:
        print("No new leads to score.")

In [8]:
X_TEST = pd.read_csv("/Users/ram/Desktop/Machine Learning Models/Leadscoring/X_test.csv")

In [10]:
Y_test = pd.read_csv("/Users/ram/Desktop/Machine Learning Models/Leadscoring/Y_test.csv")

In [9]:
PRED = LOAD_MODEL.predict(X_TEST)

In [12]:
accuracy = (PRED == Y_test.values.flatten()).mean()
accuracy

np.float64(0.866692426584235)